In [ ]:
pip install torch
python ejercicio4_grid_hparams.py texto.txt

In [ ]:
import math
import sys
import time
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# -----------------------------
# Datos
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    return text.replace("\r\n", "\n").replace("\r", "\n")

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

In [ ]:
# -----------------------------
# Mini-Transformer causal
# -----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.drop(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.drop(out)
        return out


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int, n_head: int, n_layer: int, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.seq_len
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

In [ ]:
# -----------------------------
# Train / Eval
# -----------------------------
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B*T, V), y.view(B*T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B*T, V), y.view(B*T))
        total += loss.item()
        n += 1
    return total / max(1, n)


@dataclass
class GridConfig:
    seq_len: int = 128
    n_head: int = 4
    dropout: float = 0.1
    epochs: int = 3
    device: str = "cpu"
    # grid
    lrs: Tuple[float, ...] = (1e-4, 3e-4, 1e-3)
    batch_sizes: Tuple[int, ...] = (16, 32, 64)
    n_layers: Tuple[int, ...] = (2, 4)
    n_embds: Tuple[int, ...] = (128, 256)


def run_grid(text_path: str, cfg: GridConfig):
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    data_ids = encode(text, stoi)

    n = len(data_ids)
    split = int(0.9 * n)
    train_ids = data_ids[:split]
    val_ids = data_ids[split:]

    train_ds = CharWindowDataset(train_ids, cfg.seq_len)
    val_ds = CharWindowDataset(val_ids, cfg.seq_len)

    device = torch.device(cfg.device)
    vocab_size = len(stoi)

    results: List[Dict[str, Any]] = []

    for lr in cfg.lrs:
        for bs in cfg.batch_sizes:
            for nl in cfg.n_layers:
                for ne in cfg.n_embds:
                    # Evitar configuraciones muy pesadas en CPU si quieres:
                    # if cfg.device == "cpu" and ne == 256 and nl == 4 and bs == 64:
                    #     continue

                    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True)
                    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, drop_last=True)

                    model = MiniTransformerLM(
                        vocab_size=vocab_size,
                        seq_len=cfg.seq_len,
                        n_embd=ne,
                        n_head=cfg.n_head,
                        n_layer=nl,
                        dropout=cfg.dropout
                    ).to(device)

                    optim = torch.optim.AdamW(model.parameters(), lr=lr)

                    stable = True
                    t0 = time.time()
                    try:
                        for _ in range(cfg.epochs):
                            tr = train_one_epoch(model, train_loader, optim, device)
                            if not math.isfinite(tr):
                                stable = False
                                break
                        vl = eval_loss(model, val_loader, device) if stable else float("inf")
                    except Exception:
                        stable = False
                        vl = float("inf")
                    t1 = time.time()

                    ppl = math.exp(vl) if (stable and vl < 20) else float("inf")
                    results.append({
                        "lr": lr,
                        "batch": bs,
                        "layers": nl,
                        "embd": ne,
                        "val_loss": vl,
                        "val_ppl": ppl,
                        "time_s": round(t1 - t0, 2),
                        "stable": stable
                    })

                    print(f"lr={lr:.0e} bs={bs:2d} layers={nl} embd={ne:3d} | stable={stable} | val_loss={vl:.4f} | time={t1-t0:.1f}s")

    # ordenar por val_loss
    results_sorted = sorted(results, key=lambda r: r["val_loss"])
    return results_sorted

In [ ]:
if __name__ == "__main__":

    cfg = GridConfig(
        seq_len=128,
        n_head=4,
        dropout=0.1,
        epochs=3,
        device="cpu",  # cambia a "cuda" si tienes GPU
        lrs=(1e-4, 3e-4, 1e-3),
        batch_sizes=(16, 32, 64),
        n_layers=(2, 4),
        n_embds=(128, 256),
    )

    results = run_grid("texto.txt", cfg)

    print("\n" + "=" * 90)
    print("TOP 10 configuraciones (menor val_loss):")
    for r in results[:10]:
        print(r)

    print("\n" + "=" * 90)
    print("Peores (incluye inestables):")
    for r in results[-5:]:
        print(r)